# Lab 05 - Chunking, Embeddings und Vektorsuche (Teil V)

Das technische Fundament von Tag 2. Sie entwickeln zuerst die Intuition für
Embeddings und Cosine-Similarity, bauen dann drei Chunking-Strategien selbst
und messen, welche das beste Retrieval auf dem Gold-Set liefert. Ergebnis:
Chunking ist kein Detail, sondern eine Stellschraube mit messbarem Effekt.

## 1. Embedding- und Cosine-Intuition

Ein Embedding bildet Text auf einen Vektor ab, sodass ähnliche Bedeutung nahe
beieinander liegt. Bei L2-normierten Vektoren ist das Skalarprodukt direkt die
Cosine-Similarity. Wir prüfen das an wenigen Sätzen aus der Domäne.

In [ ]:
import numpy as np

from common import embeddings as E
from common.corpus import load_corpus
from common.goldset import load_queries, qrels
from common import retrieval as R

print("Embedding-Modell:", E.active_model())

saetze = [
    "Der Dauerbetriebsdruck beträgt 350 bar.",
    "Im Dauerbetrieb sind höchstens 350 bar zulässig.",   # nah an Satz 1
    "Das erste Öl wird nach 50 Betriebsstunden gewechselt.",
    "Schutzbrille und ölbeständige Handschuhe tragen.",   # ganz anderes Thema
]
M = E.embed(saetze)
sim = E.cosine(M, M)
print("\nCosine-Matrix:")
for i, row in enumerate(sim):
    print(" ", " ".join(f"{v:5.2f}" for v in row))
print("\nSatz 1 vs 2 (gleiche Aussage):", round(float(sim[0, 1]), 3))
print("Satz 1 vs 4 (fremdes Thema):  ", round(float(sim[0, 3]), 3))

## 2. Drei Chunking-Strategien

Wir erzeugen jeweils `retrieval.Chunk`-Objekte, damit wir denselben Dense-Index
und dieselben Metriken wiederverwenden können. doc_id und acl bleiben erhalten.

In [ ]:
import re

from langchain_text_splitters import RecursiveCharacterTextSplitter

docs = load_corpus()


def chunk_absatz(docs, max_chars=550):
    """Strategie A: absatzweise packen (der Default aus common.retrieval)."""
    return R.chunk_corpus(docs, max_chars=max_chars)


def chunk_sliding(docs, size=400, overlap=80):
    """Strategie B: zeichenbasiertes Sliding-Window mit Überlappung."""
    out = []
    for d in docs:
        text = d.text
        step = max(1, size - overlap)
        i = idx = 0
        while i < len(text):
            piece = text[i:i + size].strip()
            if piece:
                out.append(R.Chunk(f"{d.doc_id}#s{idx}", d.doc_id, d.title, piece, d.acl))
                idx += 1
            i += step
    return out


def chunk_recursive(docs, size=400, overlap=60):
    """Strategie C: rekursiver Splitter (langchain), respektiert Trennzeichen."""
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=size, chunk_overlap=overlap,
        separators=["\n\n", "\n", ". ", " ", ""])
    out = []
    for d in docs:
        for idx, piece in enumerate(splitter.split_text(d.text)):
            piece = piece.strip()
            if piece:
                out.append(R.Chunk(f"{d.doc_id}#r{idx}", d.doc_id, d.title, piece, d.acl))
    return out


strategien = {
    "absatz":    chunk_absatz(docs),
    "sliding400": chunk_sliding(docs),
    "recursive400": chunk_recursive(docs),
}
for name, ch in strategien.items():
    laengen = [len(c.text) for c in ch]
    print(f"  {name:14s}: {len(ch):3d} Chunks, Median {int(np.median(laengen)):3d} Zeichen")

## 3. Retrieval-Qualität je Strategie messen

Für jede Strategie bauen wir einen Dense-Index, beantworten alle Gold-Queries,
reduzieren auf Dokumentebene und werten mit ranx aus. So wird Chunking
vergleichbar statt Geschmackssache.

In [ ]:
from ranx import Qrels, Run, evaluate

gold = Qrels(qrels())
queries = load_queries()
ergebnisse = []
laeufe = {}
for name, chunks in strategien.items():
    dense = R.DenseRetriever(chunks)
    run = {}
    for q in queries:
        doc_scores = R.collapse_to_docs(dense.search(q.text, k=10))
        run[q.qid] = {doc: sc for doc, sc in doc_scores}
    laeufe[name] = run
    m = evaluate(gold, Run(run, name=name), ["ndcg@5", "recall@5", "mrr"])
    ergebnisse.append({"strategie": name, **{k: round(float(v), 3) for k, v in m.items()}})

import pandas as pd

df = pd.DataFrame(ergebnisse).set_index("strategie")
print(df)

## 4. Visualisierung

In [ ]:
import matplotlib
matplotlib.use("Agg")  # in Jupyter inline; als Skript ohne Display
import matplotlib.pyplot as plt

ax = df.plot(kind="bar", figsize=(8, 4), rot=0)
ax.set_ylabel("Score")
ax.set_title("Retrieval-Qualität nach Chunking-Strategie")
ax.legend(loc="lower right")
plt.tight_layout()
plt.savefig("media_lab05_chunking.png", dpi=110)
print("Diagramm gespeichert: media_lab05_chunking.png")

## Aufgabe 1: Fenstergröße variieren

Lassen Sie `chunk_sliding` mit size 200, 400 und 800 laufen (overlap 0). Messen
Sie nDCG@5. Sehr kleine Chunks zerreißen Zusammenhänge, sehr große verdünnen
das Signal. Wo liegt das Optimum für diesen Korpus?

In [ ]:
# Ihre Lösung:

### Lösung

In [ ]:
for size in (200, 400, 800):
    chunks = chunk_sliding(docs, size=size, overlap=0)
    dense = R.DenseRetriever(chunks)
    run = {q.qid: {d: s for d, s in R.collapse_to_docs(dense.search(q.text, 10))}
           for q in queries}
    ndcg = float(evaluate(gold, Run(run), ["ndcg@5"]))
    print(f"size={size:3d}: {len(chunks):3d} Chunks, nDCG@5={ndcg:.3f}")

## Aufgabe 2: Welche Query schwankt am stärksten?

Vergleichen Sie pro Query den nDCG@5 zwischen 'absatz' und 'sliding400'. Welche
Frage gewinnt oder verliert am meisten durch die Strategie, und warum (kurze
gegen lange Antwortpassage)?

In [ ]:
# Ihre Lösung:

### Lösung

In [ ]:
for q in queries:
    a = float(evaluate(Qrels({q.qid: qrels()[q.qid]}), Run({q.qid: laeufe["absatz"][q.qid]}), ["ndcg@5"]))
    b = float(evaluate(Qrels({q.qid: qrels()[q.qid]}), Run({q.qid: laeufe["sliding400"][q.qid]}), ["ndcg@5"]))
    diff = a - b
    if abs(diff) > 0.05:
        print(f"  {q.qid} {q.text[:42]:42s} absatz-sliding = {diff:+.3f}")

## Diskussion

- Welcher Parameter hatte den größten Effekt: Strategie, Fenstergröße oder
  Überlappung?
- Embeddings sind sprachgebunden. Warum passt hier ein mehrsprachiges Modell,
  und woran würden Sie ein besser passendes erkennen?
- Cosine misst Ähnlichkeit, nicht Korrektheit. Wo kann ein semantisch naher,
  aber faktisch falscher Chunk gefährlich werden? (Vorschau auf Teil X.)

## Bonus: lokale gegen OpenAI-Embeddings

Bisher lief alles lokal (sentence-transformers, 384-dim). `common/embeddings.py`
schaltet per `LAB_EMBED_BACKEND=openai` (oder pro Aufruf `backend="openai"`) auf
die OpenAI-API um (`text-embedding-3-small`, 1536-dim). Wir halten die Chunking-
Strategie fest (`absatz`) und vergleichen nur die Embeddings gegen dieselben
Gold-Queries.

Voraussetzung: gültiger `OPENAI_API_KEY` in `.env` und `openai>=1.52`.

In [ ]:
def dense_run(chunks, backend):
    """Dense-Retrieval-Run mit wählbarem Embedding-Backend ('local' | 'openai')."""
    mat = E.embed([c.for_embedding() for c in chunks], backend=backend)  # (n, d), L2-normiert
    run = {}
    for q in queries:
        scores = mat @ E.embed(q.text, backend=backend)[0]
        top = [(chunks[i], float(scores[i])) for i in np.argsort(scores)[::-1][:10]]
        run[q.qid] = {doc: sc for doc, sc in R.collapse_to_docs(top)}
    return run

chunks_ab = strategien["absatz"]
runs = {b: dense_run(chunks_ab, b) for b in ("local", "openai")}

vergleich = [
    {"embeddings": b,
     **{k: round(float(v), 3)
        for k, v in evaluate(gold, Run(runs[b], name=b), ["ndcg@5", "recall@5", "mrr"]).items()}}
    for b in ("local", "openai")
]
print(pd.DataFrame(vergleich).set_index("embeddings"))

Zur Anschauung die Top-3-Dokumente je Backend für einige Queries:

In [ ]:
for q in queries[:5]:
    print(f"\n{q.text}")
    print(f"  local : {list(runs['local'][q.qid])[:3]}")
    print(f"  openai: {list(runs['openai'][q.qid])[:3]}")

**Beobachtung:** mehr Dimensionen heißen nicht automatisch besser für jede
Frage, aber OpenAI-Embeddings trennen semantische Umschreibungen meist sauberer.
Der Preis: pro Query ein API-Call und Netzabhängigkeit. Fürs Offline-Training
bleibt `local` die Standardwahl, für Qualitätsvergleiche ist `openai` einen
Schalter entfernt.